In [7]:
import numpy as np
import pandas as pd
import wave
import math
import struct
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Activation

In [8]:
# Prepare a dummy data for simulating musical notes
notes_freqs = {
    'A': 440.0, 'B': 493.88, 'C': 261.63, 'D': 293.66, 'E': 393.63,
    'F': 349.23, 'G': 392.0
}
notes_freqs

{'A': 440.0,
 'B': 493.88,
 'C': 261.63,
 'D': 293.66,
 'E': 393.63,
 'F': 349.23,
 'G': 392.0}

In [9]:
notes = list(notes_freqs.keys())
notes

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [10]:
note_to_int = { note : i for i, note in enumerate(notes)}
note_to_int

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}

In [11]:
int_to_note = { i : note for i, note in enumerate(notes)}
int_to_note

{0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

In [12]:
raw_music_data = [notes[np.random.randint(0,7)] for i in range(1000)]
raw_music_data

['A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'E',
 'C',
 'B',
 'E',
 'G',
 'A',
 'G',
 'F',
 'F',
 'G',
 'G',
 'C',
 'G',
 'C',
 'D',
 'B',
 'C',
 'D',
 'B',
 'G',
 'F',
 'C',
 'B',
 'A',
 'E',
 'A',
 'F',
 'B',
 'F',
 'C',
 'A',
 'F',
 'F',
 'B',
 'A',
 'B',
 'E',
 'B',
 'G',
 'G',
 'F',
 'D',
 'A',
 'G',
 'B',
 'E',
 'A',
 'F',
 'A',
 'B',
 'D',
 'A',
 'B',
 'E',
 'B',
 'D',
 'G',
 'F',
 'E',
 'G',
 'C',
 'B',
 'A',
 'E',
 'A',
 'A',
 'F',
 'A',
 'G',
 'C',
 'G',
 'A',
 'B',
 'C',
 'F',
 'B',
 'F',
 'C',
 'E',
 'G',
 'F',
 'C',
 'D',
 'D',
 'A',
 'G',
 'A',
 'F',
 'E',
 'D',
 'D',
 'D',
 'D',
 'E',
 'B',
 'F',
 'D',
 'E',
 'D',
 'A',
 'A',
 'G',
 'A',
 'C',
 'F',
 'C',
 'F',
 'G',
 'B',
 'C',
 'C',
 'B',
 'A',
 'D',
 'D',
 'G',
 'D',
 'A',
 'D',
 'C',
 'G',
 'A',
 'G',
 'F',
 'G',
 'B',
 'E',
 'F',
 'E',
 'E',
 'F',
 'D',
 'C',
 'A',
 'A',
 'F',
 'D',
 'A',
 'C',
 'D',
 'D',
 'A',
 'E',
 'E',
 'B',
 'B',
 'D',
 'F',
 'C',
 'B',
 'F',
 'B',
 'D',
 'D',
 'B',
 'F',
 'E',
 'G',
 'E',
 'G',
 'D'

#### Data Preparation

In [13]:
seq_length = 3
network_input = []
network_output = []


for i in range(len(raw_music_data) - seq_length):
    seq_in = raw_music_data[i: i+seq_length]
    seq_out = raw_music_data[i+seq_length]
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])
    print(seq_in,'-->',seq_out)

['A', 'A', 'A'] --> A
['A', 'A', 'A'] --> A
['A', 'A', 'A'] --> A
['A', 'A', 'A'] --> E
['A', 'A', 'E'] --> C
['A', 'E', 'C'] --> B
['E', 'C', 'B'] --> E
['C', 'B', 'E'] --> G
['B', 'E', 'G'] --> A
['E', 'G', 'A'] --> G
['G', 'A', 'G'] --> F
['A', 'G', 'F'] --> F
['G', 'F', 'F'] --> G
['F', 'F', 'G'] --> G
['F', 'G', 'G'] --> C
['G', 'G', 'C'] --> G
['G', 'C', 'G'] --> C
['C', 'G', 'C'] --> D
['G', 'C', 'D'] --> B
['C', 'D', 'B'] --> C
['D', 'B', 'C'] --> D
['B', 'C', 'D'] --> B
['C', 'D', 'B'] --> G
['D', 'B', 'G'] --> F
['B', 'G', 'F'] --> C
['G', 'F', 'C'] --> B
['F', 'C', 'B'] --> A
['C', 'B', 'A'] --> E
['B', 'A', 'E'] --> A
['A', 'E', 'A'] --> F
['E', 'A', 'F'] --> B
['A', 'F', 'B'] --> F
['F', 'B', 'F'] --> C
['B', 'F', 'C'] --> A
['F', 'C', 'A'] --> F
['C', 'A', 'F'] --> F
['A', 'F', 'F'] --> B
['F', 'F', 'B'] --> A
['F', 'B', 'A'] --> B
['B', 'A', 'B'] --> E
['A', 'B', 'E'] --> B
['B', 'E', 'B'] --> G
['E', 'B', 'G'] --> G
['B', 'G', 'G'] --> F
['G', 'G', 'F'] --> D
['G', 'F',

In [14]:
n_paterns = len(network_input)
n_paterns

997

In [15]:
x = np.reshape(network_input,(n_paterns, seq_length, 1))
x

array([[[0],
        [0],
        [0]],

       [[0],
        [0],
        [0]],

       [[0],
        [0],
        [0]],

       ...,

       [[6],
        [0],
        [3]],

       [[0],
        [3],
        [2]],

       [[3],
        [2],
        [5]]], shape=(997, 3, 1))

In [16]:
from keras.utils import to_categorical

In [17]:
y = to_categorical(network_output)
y.shape

(997, 7)

In [18]:
y

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], shape=(997, 7))

#### Build the Model

In [22]:
from tensorflow.keras.layers import Input

# Now you can define your model
model = Sequential()
model.add(Input(shape=(3, 1)))
model.add(GRU(256))
model.add(Dense(512, activation='relu'))
model.add(Dense(7, activation='softmax'))

In [23]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_3 (GRU)                     │ (None, 256)            │       198,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 334,087 (1.27 MB)

 Trainable params: 334,087 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.compile(
    loss='categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)

In [25]:
model.fit(x, y, epochs=100, batch_size=10)

Epoch 1/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.1494 - loss: 1.9683
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1454 - loss: 1.9507
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1424 - loss: 1.9489
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1163 - loss: 1.9466
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1464 - loss: 1.9461
Epoch 6/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1625 - loss: 1.9427
Epoch 7/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1665 - loss: 1.9414
Epoch 8/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1565 - loss: 1.9415
Epoch 9/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1725 - loss: 1.9381
Epoch 10/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1545 - loss: 1.9415
Epoch 11/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1575 - loss: 1.9372
Epoch 12/100
100/100 ━━━━━━━━━━━━━━━━━━━━

#### Generate New Melody Sequence

In [27]:
start_index = np.random.randint(0, len(network_output))
pattern = network_input[start_index]
pattern

[6, 1, 6]

In [30]:
generated_melody = []
for i in range(50):
    x_input = np.reshape(pattern, (1, len(pattern), 1))
    pred = model.predict(x_input, verbose=False)
    index = np.argmax(pred)
    result = int_to_note[index]
    generated_melody.append(result)
    pattern.append(index)
    pattern = pattern[1:len(pattern)]

In [31]:
generated_melody

['C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D',
 'C',
 'A',
 'C',
 'D']

In [ ]:

with wave.open('my_music.wav', 'w') as wav_file:

    wav_file.setparams((1, 2, 44100, 0, 'NONE', 'not compressed'))
    
    for note in generated_melody:
        freq = notes_freqs[note]
        num_samples = int(0.5 * 44100) 
        
        for i in range(num_samples):
            t = float(i) / 44100
            value = int(32767 * 0.5 * math.sin(2 * math.pi * freq * t))
            
            data = struct.pack('<h', value)
            wav_file.writeframes(data)